# 04 — Latent Space Visualisation
UMAP/t-SNE of latent codes, coloured by ACDC pathology class.

In [3]:
import os
os.chdir('/workspace/ACDC-demo')


In [4]:
import sys; sys.path.insert(0, '..')
import json
import torch
from pathlib import Path
from torch_geometric.loader import DataLoader
from src.models.encoder import GraphEncoder
from src.training.train_main import CardiacGraphDataset
from src.utils.visualization import plot_latent_space
from src.utils.acdc_loader import PATHOLOGY_CLASSES

device = torch.device('cpu')
ckpt = torch.load('checkpoints/main/best_model.pt', map_location=device, weights_only=False)
cfg = ckpt['cfg']

encoder = GraphEncoder(hidden_dim=cfg['hidden_dim'], latent_dim=cfg['latent_dim'], n_layers=cfg['n_encoder_layers'])
encoder.load_state_dict(ckpt['encoder'])
encoder.eval()

with open('data/graphs/manifest.json') as f:
    manifest = json.load(f)
entries = [e for e in manifest if e['frame'] == 'ed' and e['struct'] == 'lv']
ds = CardiacGraphDataset('data/graphs', entries, 'lv')
loader = DataLoader(ds, batch_size=8)

import numpy as np
latents, labels = [], []
with torch.no_grad():
    for batch in loader:
        z = encoder.encode(batch)
        latents.append(z.numpy())
        labels.append(batch.y.numpy())
latents = np.concatenate(latents)
labels = np.concatenate(labels)
label_names = list(PATHOLOGY_CLASSES.keys())

In [ ]:
fig = plot_latent_space(latents, labels, label_names=label_names, method='umap',
                         out_path='report/figures/latent_space_umap.png')
fig.show()